In [ ]:
! pip show transformers

In [ ]:
! pip install -U bitsandbytes

In [ ]:
! pip install pillow

In [ ]:
import re

In [ ]:
from transformers import BitsAndBytesConfig

quant_config = BitsAndBytesConfig(
                load_in_8bit=True
            )

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# model_id = "google/gemma-3-1b-it"
model_id = "tiiuae/falcon-7b-instruct"

# model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=quant_config, device_map="auto")


model = AutoModelForCausalLM.from_pretrained(
    model_id, torch_dtype=torch.bfloat16
).eval()

tokenizer = AutoTokenizer.from_pretrained(model_id, padding_side="left")

/opt/miniconda3/envs/fm_gemma/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00, 64.18it/s]


In [2]:
# messages = [
#     [
#         {
#             "role": "system",
#             "content": [{"type": "text", "text": "You are a helpful assistant."},]
#         },
#         {
#             "role": "user",
#             "content": [{"type": "text", "text": "Write a poem on Hugging Face, the company"},]
#         },
#     ],
# ]
# inputs = tokenizer.apply_chat_template(
#     messages,
#     add_generation_prompt=True,
#     tokenize=True,
#     return_dict=True,
#     return_tensors="pt",
# ).to(model.device)


# with torch.inference_mode():
#     outputs = model.generate(**inputs, max_new_tokens=64)

# generated_tokens = outputs[0, inputs["input_ids"].shape[1]:]
# outputs1 = tokenizer.batch_decode(generated_tokens)

# outputs2 = tokenizer.batch_decode(outputs)


In [ ]:
outputs2

In [ ]:
generated_tokens

In [ ]:
outputs1 = tokenizer.decode(generated_tokens)
outputs1

In [2]:
line = {
    "subj": "Oral",
    # "subj": "Herlyn Espinal",
    # "question": "What is Herlyn Espinal's occupation?"
    "question": "What is Oral the capital of?"
}

system = {
    "role": "system",
    "content": "You'll be given a question about the article and answer it with a one word. Answer the [Question]. "
}
user = {
    "role": "user",
    "content": "This article is about %s. [Question]: %s [Answer]:" % (
        line["subj"], line["question"])
}
messages = [system, user]

# inputs = tokenizer.apply_chat_template(
#     messages,
#     add_generation_prompt=True,
#     tokenize=True,
#     return_dict=True,
#     return_tensors="pt",
# ).to(model.device)

# with torch.inference_mode():
#     outputs = model.generate(**inputs, max_new_tokens=20, do_sample=True)

# # generated_tokens = outputs[0, inputs["input_ids"].shape[1]:]
# decoded = tokenizer.batch_decode(outputs)
# # outputs2 = tokenizer.decode(generated_tokens, skip_special_tokens=True)

encoded = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to(model.device)
generated_ids = model.generate(encoded, max_new_tokens=20, do_sample=True)
decoded = tokenizer.batch_decode(generated_ids)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


In [3]:
tokenizer.decode(encoded[0])

"You'll be given a question about the article and answer it with a one word. Answer the [Question].\n\nUser: This article is about Oral. [Question]: What is Oral the capital of? [Answer]:\n\nAssistant:"

In [4]:
tokenizer.eos_token

'<|endoftext|>'

In [5]:
decoded

["You'll be given a question about the article and answer it with a one word. Answer the [Question].\n\nUser: This article is about Oral. [Question]: What is Oral the capital of? [Answer]:\n\nAssistant: Hungary\nUser <|endoftext|>"]

In [16]:
result = decoded[0].split('<start_of_turn>model')[-1]
result = result.replace("<end_of_turn>", "")
result.strip()

'Rome'

In [ ]:
# decoded=["<s> [INST] <<SYS>>\nYou'll be given a question about the article and answer it with a one word. Answer the [Question]. \n<</SYS>>\n\nThis article is about Paris. [Question]: What is Paris's occupation? [Answer]: [/INST]  Mayor</s>"]

In [ ]:
result = re.search(".*\[\/INST\](.*)</s>", decoded[0])
print(result.groups())
if not result:
    answer = ""
elif isinstance(result, str):
    print("this one")
    answer = result.strip()
else:
    print("that one")
    answer = result.groups()[0].strip()
print(answer)

In [ ]:
generated_tokens = outputs[0, inputs["input_ids"].shape[1]:]
outputs1 = tokenizer.decode(generated_tokens)
outputs1

In [ ]:
answer = decoded[0].split("<start_of_turn>model")[1]
answer = answer.split("<end_of_turn>")[0]
answer

In [ ]:
tokenizer.eos_token

In [ ]:
import torch
import numpy as np
from transformers import (
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    GenerationConfig,
    LlamaTokenizer,
    T5TokenizerFast,
    GemmaTokenizerFast,
    PreTrainedTokenizerFast,
    AutoProcessor
)
import json

In [ ]:
MODEL_NAME_OR_PATH = "google/gemma-3-1b-it"
# MODEL_NAME_OR_PATH = "tiiuae/falcon-7b-instruct"
# MODEL_NAME_OR_PATH = "meta-llama/Meta-Llama-3-8B-Instruct"

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if "gemma" in MODEL_NAME_OR_PATH:
    model =  AutoModelForCausalLM.from_pretrained(MODEL_NAME_OR_PATH, torch_dtype=torch.bfloat16).to(
                    device
                )
else:
    model =  AutoModelForCausalLM.from_pretrained(MODEL_NAME_OR_PATH, torch_dtype=torch.float16).to(
                    device
                )
type(model)

In [ ]:
NUM_BEAMS = 1
MAX_ANSWER_LENGTH = 10
DEF_TEMPLATE_TO_USE = "query_in_response"
DEF_INSTRUCTION = "Complete the fact in as few words as possible"

TEMPLATES = {
    "query_in_instructions": (
        "Below is an instruction that describes a task. "
        "Write a response that appropriately completes the request.\n\n"
        "### Instruction:\n{}: {}\n\n### Response:"
    ),
    "query_in_response": (
        "Below is an instruction that describes a task. "
        "Write a response that appropriately completes the request.\n\n"
        "### Instruction:\n{}\n\n### Response: {}"
    ),
    "query_in_input": (
        "Below is an instruction that describes a task. "
        "Write a response that appropriately completes the request.\n\n"
        "### Instruction:\n{}\n\n### Input:\n{}\n\n### Response:"
    ),
}

def prepare_prompt(query, model_name_or_path, instruction, tokenizer, template=None):
    if "alpaca" in model_name_or_path:
        instruction = instruction
        template = TEMPLATES[template]
        return template.format(instruction, query)
    elif "instruct" in model_name_or_path in model_name_or_path:
        return "{}\n{}".format(instruction, query)
    elif "chat" in model_name_or_path:
        return "[INST] {}: {} [/INST] ".format(instruction, query)
    elif "gemma-3-1b" in model_name_or_path:
        # system = "Complete the fact in as few words as possible"
        messages = [{
                "role": "system",
                "content": instruction
            },{
                "role": "user",
                "content": query        
            }]
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
    elif "gemma-3-12b" in model_name_or_path:
        # system = "Complete the fact in as few words as possible"
        messages = [{
                "role": "system",
                "content": [{"type": "text", "text": instruction}]
            },{
                "role": "user",
                "content": [{"type": "text", "text": query}]        
            }]
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
    else:
        return query

In [ ]:
get_sequence = {
    # Ignore the prompt.
    LlamaTokenizer: lambda seq, input_ids: seq[input_ids.shape[1] :].cpu().tolist(),
    PreTrainedTokenizerFast: lambda seq, input_ids: seq[input_ids.shape[1] :]
    .cpu()
    .tolist(),
    # Ignore the BOS token.
    GemmaTokenizerFast: lambda seq, input_ids: seq[input_ids.shape[1] :].cpu().tolist(),
    T5TokenizerFast: lambda seq, _: seq.cpu().tolist()[1:],
}
ids_to_ignore = {
    # Ignore BOS, EOS.
    LlamaTokenizer: [1, 2],
    # Ignore EOS.
    T5TokenizerFast: [1],

    # Add this line for Gemma (BOS=2, EOS=1, <end_of_turn>=106)
    GemmaTokenizerFast: [1, 2, 106],

    # Ignore EOS.
    PreTrainedTokenizerFast: [11],
}
# Token id of a full stop when not at the beggining of a word so it could be
# different than tokenizer.tokens_to_ids(tokenizer.tokenize('.')).
full_stop = {LlamaTokenizer: 29889, T5TokenizerFast: 5, GemmaTokenizerFast: 236761, PreTrainedTokenizerFast: 25, }

In [ ]:
def get_scores(model_output, input_ids, prompt, query, tokenizer):
    """Assumes num_beam=1. Gets the token scores for every token that is not BOS, EOS or fullstop,
    gets the first non-the token score and computes pplx."""
    sequence = get_sequence[type(tokenizer)](model_output["sequences"][0], input_ids)
    raw_answer = tokenizer.decode(sequence).strip()
    print(f"{raw_answer=}")
    assert len(sequence) == len(model_output["scores"])
    token_scores = []
    trimmed_sequence = []
    for idx, score in zip(sequence, model_output["scores"]):
        if idx not in ids_to_ignore[type(tokenizer)]:
            token_scores.append(torch.softmax(score, 1)[:, idx].cpu().item())
            trimmed_sequence.append(idx)
    if trimmed_sequence and trimmed_sequence[-1] == full_stop[type(tokenizer)]:
        token_scores = token_scores[:-1]
        trimmed_sequence = trimmed_sequence[:-1]
    answer = tokenizer.decode(trimmed_sequence).strip()
    words = answer.split()
    if (
        not token_scores
        or not words
        or (
            (len(token_scores) == 1 or len(words) == 1)
            and words[0] in ["the", "a", "an"]
        )
    ):
        print(
            "Warning: Empty generation. input_ids={}, output_sequence={}".format(
                input_ids, sequence
            )
        )
        return "", [], 0, float("inf")
    first_token_score = (
        token_scores[1] if words[0] in ["the", "a", "an"] else token_scores[0]
    )
    perplexity = np.exp(-np.mean(np.log(token_scores)))

    return answer, token_scores, first_token_score, perplexity

In [ ]:
def get_generation_config(tokenizer):
    print(type(tokenizer))
    if tokenizer.pad_token_id is None:
        print("No PAD token by default")
        tokenizer.pad_token = tokenizer.eos_token

    return GenerationConfig(
        max_new_tokens=50,
        num_beams=NUM_BEAMS,
        do_sample=False,
        output_hidden_states=False,
        output_scores=False,
        num_return_sequences=NUM_BEAMS,
        return_dict_in_generate=True,
        pad_token_id=tokenizer.pad_token_id,
    )


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME_OR_PATH, use_fast=True
    )

config = get_generation_config(tokenizer)
print(config)

In [ ]:
# MODEL_NAME_OR_PATH = "google/gemma-3-1b-it"
MODEL_NAME_OR_PATH = "tiiuae/falcon-7b-instruct"
# MODEL_NAME_OR_PATH = "meta-llama/Meta-Llama-3-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME_OR_PATH, use_fast=True
    )

In [ ]:
tokenizer.padding_side

In [ ]:
tokenizer.name_or_path

In [ ]:
tokenizer.eos_token

In [ ]:
tokenizer.pad_token

In [ ]:
tokenizer.pad_token_id

In [ ]:
query = "Friedrich Theodor Vischer took up work located in"
with torch.no_grad():
    prompt = prepare_prompt(
        query, MODEL_NAME_OR_PATH, DEF_INSTRUCTION, tokenizer, DEF_TEMPLATE_TO_USE
    )
    print(f"{prompt=}")
    if "gemma" in MODEL_NAME_OR_PATH:
        input_ids = tokenizer.encode(prompt, return_tensors="pt", add_special_tokens=False).to(device)
    else:
        input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    print(f"{tokenizer.decode(input_ids[0])=}")
    
    # debug
    print(f"{tokenizer(prompt, return_tensors="pt", add_special_tokens=False)=}")
    print(f"{tokenizer.decode(tokenizer(prompt, return_tensors="pt", add_special_tokens=False)["input_ids"][0])=}")
    # prompt = tokenizer.apply_chat_template(
    #     messages,
    #     tokenize=False,
    #     add_generation_prompt=True
    # )

    # debug block
    # messages = [{
    #             "role": "system",
    #             "content": [{"type": "text", "text": DEF_INSTRUCTION}]
    #         },{
    #             "role": "user",
    #             "content": [{"type": "text", "text": query}]        
    #         }]
    # input_ids2 = tokenizer.apply_chat_template(
    #     messages,
    #     add_generation_prompt=True,
    #     tokenize=True,
    #     # return_dict=True,
    #     return_tensors="pt",
    # ).to(device)
    # print(f"{tokenizer.decode(input_ids2[0])=}")
    # print(f"{tokenizer.decode(tokenizer.encode("Hello"))=}")
    
    model_output = model.generate(
        input_ids, generation_config=config, output_scores=True
    )
    full_generation = tokenizer.decode(model_output["sequences"][0])
    print(f"{full_generation=}")

answer, token_scores, first_token_score, perplexity = get_scores(
    model_output, input_ids, prompt, query, tokenizer
)

In [ ]:
# tokenizer = AutoTokenizer.from_pretrained(
#             "google/gemma-3-12b-it", use_fast=True
#         )
tokenizer = AutoProcessor.from_pretrained("google/gemma-3-12b-it", use_fast=True)
print(prepare_prompt(
        query, "google/gemma-3-12b-it", DEF_INSTRUCTION, tokenizer, DEF_TEMPLATE_TO_USE
    ))

# --- Step 1: Check the Chat Template ---
print("--- 1. Does the template string start with a BOS token? ---")
template_starts_with_bos = tokenizer.chat_template.lstrip().startswith('{{ bos_token }}')
print(f"Answer: {template_starts_with_bos}")
# print("Template:\n", tokenizer.chat_template) # Uncomment to see the full template

print("\n" + "="*50 + "\n")

# --- Step 2: Check the Tokenizer's Default Behavior ---
print("--- 2. Does the tokenizer add a BOS token by default? ---")
# Tokenize a simple word. The tokenizer adds special tokens by default.
encoded = tokenizer.encode("Hello")
# Decode only the very first token ID from the output
first_token_decoded = tokenizer.decode(encoded[0])
print(f"Answer: Yes, the first token it adds is: '{first_token_decoded}'")

print("\n" + "="*50 + "\n")

# --- Step 3: See the Collision ---
print("--- 3. Simulating the double-token effect ---")
messages = [{"role": "user", "content": "Hi"}]
# We use tokenize=False to see what the template *alone* produces
string_from_template = tokenizer.apply_chat_template(messages, tokenize=False)
print(f"Output from template only: \n{string_from_template}\n")

# Now, let's tokenize that string
final_tokens = tokenizer.tokenize(string_from_template)
print(f"Result of tokenizing that string (first 5 tokens): \n{final_tokens[:5]}")



In [ ]:
answer

In [ ]:
type(model)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-1b-it")

# Print the EOS token
print(f"The EOS token is: {tokenizer.eos_token}")

# You can also check its ID
print(f"The EOS token ID is: {tokenizer.eos_token_id}")

In [ ]:
text_to_tokenize = "Vienna<end_of_turn>"

# Use the .tokenize() method to see the token strings
tokens = tokenizer.tokenize(text_to_tokenize)

# Use the .encode() method to see the corresponding token IDs
token_ids = tokenizer.encode(text_to_tokenize)

print(f"Original string: '{text_to_tokenize}'")
print("---")
print(f"Tokens: {tokens}")
print(f"Token IDs: {token_ids}")

# Let's map them one-to-one for clarity
# Note: encode() adds a BOS token (ID=2) at the start, so we skip it for this mapping.
print("\n--- Breakdown ---")
for token, token_id in zip(tokens, token_ids[1:]): # Slicing token_ids[1:] to skip BOS
    print(f"'{token}'  ->  ID: {token_id}")
